# BRONCO150 als Test-Set — Zero-Shot-Evaluation

Evaluiert deine **bereits trainierten** Modelle (Baseline und +synth, je 4 Seeds)
**ohne Nachtraining** auf echten Klinik-Notizen. Gewertet wird **per-Typ-F1** für die
geteilten Klassen **Medikation** und **Diagnose** (BRONCO kennt kein `Dosis`, deshalb
ist die Gesamt-F1 hier unfair — wir schauen per Typ).

**Wichtig:**
- **Lokal** ausführen im Env `spacy-test` (mit `spacy-transformers`). BRONCO **nicht**
  nach Colab/Drive laden — die Data-Use-Agreement beschränkt die Weitergabe.
- Läuft auf CPU; 8 Modelle × ~11k Sätze dauern zusammen etwas → erst mit `MAX_DOCS` antesten.

## 1) Konfiguration — Pfade anpassen

In [ ]:
from pathlib import Path

# BRONCO CoNLL-Dateien (alle 5 Splits als EIN Test-Set)
CONLL_FILES = [
    "pfad/zu/randomSentSet1.conll",
    "pfad/zu/randomSentSet2.conll",
    "pfad/zu/randomSentSet3.conll",
    "pfad/zu/randomSentSet4.conll",
    "pfad/zu/randomSentSet5.conll",
]
BRONCO_SPACY = "meaningful_modification/data/bronco_test.spacy"

# Deine trainierten Modelle: jeweils der model-best-Ordner. PFADE/NAMEN ANPASSEN!
MODELS = {
    "baseline": [
        "related_work/models/gbert-large-seed-42-model-best",
    ],
    "+synth": [
        "meaningful_modification/models/gbert-large-syn-seed42-model-best",
    ],
}

# Nur diese (in BRONCO vorhandenen) Klassen werden gewertet
EVAL_LABELS = ["Medikation", "Diagnose"]

# Zum schnellen Antesten z.B. 300; fuer den echten Lauf None
MAX_DOCS = None

## 2) CoNLL-Format prüfen

Kurz reinschauen: In **welcher Spalte** steht der NER-Tag und **wie heißen die Typen**
(z.B. `B-MED`, `B-DIAG`)? Falls nötig `TAG_COL` / `LABEL_MAP` oben in
`meaningful_modification/scripts/bronco_to_docbin.py` anpassen.

In [ ]:
!head -20 {CONLL_FILES[0]}

## 3) BRONCO → `.spacy` konvertieren

In [ ]:
files = " ".join(CONLL_FILES)
!python meaningful_modification/scripts/bronco_to_docbin.py --conll {files} --out {BRONCO_SPACY}

## 4) Beide Modelle evaluieren

Die Schleife läuft über **beide** Bedingungen und alle Seeds und sammelt per-Typ-F1.

In [ ]:
import re
import spacy
from spacy.tokens import DocBin
from spacy.training import Example


def load_examples(nlp, spacy_path, max_docs=None):
    doc_bin = DocBin().from_disk(spacy_path)
    examples = []
    for i, gold in enumerate(doc_bin.get_docs(nlp.vocab)):
        if max_docs is not None and i >= max_docs:
            break
        examples.append(Example(nlp.make_doc(gold.text), gold))
    return examples


def evaluate_model(model_path, spacy_path, max_docs=None):
    nlp = spacy.load(model_path)
    examples = load_examples(nlp, spacy_path, max_docs)
    return nlp.evaluate(examples)


records = []
for condition, paths in MODELS.items():
    for model_path in paths:
        match = re.search(r"seed[-_]?(\d+)", model_path)
        seed = int(match.group(1)) if match else -1

        scores = evaluate_model(model_path, BRONCO_SPACY, MAX_DOCS)
        per_type = scores.get("ents_per_type", {})

        for label in EVAL_LABELS:
            s = per_type.get(label, {})
            records.append({
                "condition": condition, "seed": seed, "label": label,
                "p": s.get("p", 0.0), "r": s.get("r", 0.0), "f": s.get("f", 0.0),
            })
        print(f"[fertig] {condition} seed {seed}: "
              + ", ".join(f"{l} F1={per_type.get(l, {}).get('f', 0):.3f}" for l in EVAL_LABELS))

## 5) Auswertung: Baseline vs. +synth (mean ± std über Seeds)

In [ ]:
import pandas as pd

df = pd.DataFrame(records)

summary = (df.groupby(["label", "condition"])["f"]
             .agg(mean="mean", std="std").reset_index())

mean_tbl = summary.pivot(index="label", columns="condition", values="mean").round(4)
std_tbl  = summary.pivot(index="label", columns="condition", values="std").round(4)
mean_tbl["delta_F1 (+synth - baseline)"] = (mean_tbl.get("+synth") - mean_tbl.get("baseline")).round(4)

print("F1 (Mittelwert ueber Seeds):")
print(mean_tbl)
print("\nF1 (Std ueber Seeds):")
print(std_tbl)

df  # Rohwerte pro Modell/Seed